# YOLOv10 Edge Training and Realtime Pipeline

This notebook trains a YOLOv10 detector on a custom dataset and builds an edge-friendly realtime pipeline with async capture, GPU acceleration, adaptive throttling, and ONNX/TensorRT export for low-latency deployment.

In [ ]:
# Section 1: Import and Environment Check (CUDA)
import os, sys, time, math, json, shutil, random, pathlib
import numpy as np
import torch
import cv2

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    torch.backends.cudnn.benchmark = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Section 2: Install and Setup YOLOv10
import subprocess, sys

yolov10_dir = pathlib.Path('yolov10')
if not yolov10_dir.exists():
    !git clone https://github.com/THU-MIG/yolov10.git

# Install minimal dependencies
%pip -q install -U ultralytics onnx onnxruntime-gpu onnxruntime

# Quick test: download a sample image and run inference with YOLOv10n (PyTorch path)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # placeholder for quick sanity; we train v10 below but v8 weights let us verify pipeline quickly
img_url = 'https://ultralytics.com/images/bus.jpg'
import urllib.request
urllib.request.urlretrieve(img_url, 'bus.jpg')
res = model.predict('bus.jpg', imgsz=640, conf=0.25, verbose=False)
print('Sanity predictions:', len(res[0].boxes) if res and res[0].boxes is not None else 0)

In [ ]:
# Section 3: Dataset Preparation (YOLO format)
from pathlib import Path
from sklearn.model_selection import train_test_split
%pip -q install scikit-learn

# Assume dataset in datasets/custom with images/ and labels/ in YOLO format
root = Path('datasets/custom')
(root / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(root / 'images' / 'val').mkdir(parents=True, exist_ok=True)
(root / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
(root / 'labels' / 'val').mkdir(parents=True, exist_ok=True)

# If you have a flat images/ and labels/ directory, split it here.
all_imgs = list((root / 'images').glob('*.jpg')) + list((root / 'images').glob('*.png'))
if all_imgs:
    train_imgs, val_imgs = train_test_split(all_imgs, test_size=0.2, random_state=SEED)
    def move_split(files, split):
        for p in files:
            dst = root / 'images' / split / p.name
            if not dst.exists():
                shutil.copy2(p, dst)
            lbl = (root / 'labels' / (p.stem + '.txt'))
            if lbl.exists():
                shutil.copy2(lbl, root / 'labels' / split / lbl.name)
    move_split(train_imgs, 'train')
    move_split(val_imgs, 'val')

# Create data.yaml
classes = [
    'person','bicycle','car','motorcycle','bus','truck'  # edit for your dataset
]
(data_yaml := root / 'data.yaml').write_text('\n'.join([
    f"path: {root.resolve()}",
    "train: images/train",
    "val: images/val",
    "names:",
    *[f"  {i}: {n}" for i,n in enumerate(classes)]
]))
print('Wrote', data_yaml)

# Visual sanity-check
import matplotlib.pyplot as plt
%matplotlib inline

def show_samples(n=3):
    subset = random.sample(list((root / 'images' / 'train').glob('*.*')), min(n, len(list((root / 'images' / 'train').glob('*.*')))))
    for p in subset:
        img = cv2.imread(str(p))[:, :, ::-1]
        h, w = img.shape[:2]
        lbl = (root/'labels'/'train'/ (p.stem + '.txt'))
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                cid, cx, cy, bw, bh = map(float, line.split())
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        plt.figure(figsize=(5,5)); plt.imshow(img); plt.axis('off')

show_samples(3)

In [ ]:
# Section 4: Train YOLOv10 on Custom Dataset
from ultralytics import YOLO

# Use a YOLOv8 config as placeholder or bring YOLOv10 repo model; many users adapt YOLOv8 API to train v10 forks
model_size = 'n'  # n/s/m/l/x
epochs = 50
imgsz = 640
bs = 16

model = YOLO(f'yolov8{model_size}.pt')  # replace with YOLOv10 weights when available
results = model.train(data=str(data_yaml), epochs=epochs, imgsz=imgsz, batch=bs, device=0 if torch.cuda.is_available() else 'cpu', pretrained=True, workers=2)
print('Training complete')

In [ ]:
# Section 5: Validate and Visualize Results
val = model.val(data=str(data_yaml), imgsz=imgsz, device=0 if torch.cuda.is_available() else 'cpu')
print('mAP50-95:', val.results_dict.get('metrics/mAP50-95(B)', 'n/a'))

# Visualize predictions on a few images
pred_dir = pathlib.Path('runs/predict/val_samples')
pred_dir.mkdir(parents=True, exist_ok=True)
sample_imgs = list((root/'images'/'val').glob('*.*'))[:5]
for p in sample_imgs:
    r = model.predict(str(p), imgsz=imgsz, conf=0.25, save=True, project=str(pred_dir), name='.', exist_ok=True, verbose=False)
print('Saved sample predictions to', pred_dir)

In [ ]:
# Section 6: Export to ONNX/TensorRT (FP16/INT8)
import pathlib
import sys
import torch
from datetime import datetime

try:
    model  # noqa: F821  # trained model from earlier cells
except NameError:
    # Fallback: try to load a default trained weights if present
    try:
        from ultralytics import YOLO
        model = YOLO('runs/detect/train/weights/best.pt')
        print('Loaded model from runs/detect/train/weights/best.pt')
    except Exception as e:
        raise RuntimeError('Model is not defined and default weights not found. Train the model first.') from e

# Configure export
imgsz = 640
use_half = torch.cuda.is_available()
dynamic_axes = False  # keep static shapes for best runtime perf on edge
opset = 12

onnx_path = pathlib.Path('runs/weights/best.onnx')
onny_dir = onnx_path.parent
onny_dir.mkdir(parents=True, exist_ok=True)

print(f'Exporting to ONNX at {onnx_path} (half={use_half}, dynamic={dynamic_axes}, opset={opset})...')
start = datetime.now()
exp_path = model.export(
    format='onnx',
    imgsz=imgsz,
    simplify=True,
    dynamic=dynamic_axes,
    half=use_half,
    opset=opset,
)
elapsed = (datetime.now() - start).total_seconds()
print(f'Export complete in {elapsed:.2f}s -> {exp_path}')

# Optional: Build a TensorRT engine using trtexec (requires NVIDIA TensorRT installed)
print('\nOptional TensorRT build (run externally if you have TensorRT):')
print('trtexec --onnx="runs/weights/best.onnx" --saveEngine="runs/weights/best_fp16.trt" --fp16 --workspace=4096')


In [ ]:
# Section 7: Edge Runtime (Async Capture + ONNXRuntime Inference)
import cv2
import numpy as np
import onnxruntime as ort
import threading
import time

class LastFrameBuffer:
    def __init__(self):
        self._lock = threading.Lock()
        self._frame = None
    def put(self, frame):
        with self._lock:
            self._frame = frame
    def get(self):
        with self._lock:
            return None if self._frame is None else self._frame.copy()

class AsyncGrabber(threading.Thread):
    def __init__(self, src=0, width=None, height=None, backend=None, buffer: LastFrameBuffer=None):
        super().__init__(daemon=True)
        self.src = src
        self.width = width
        self.height = height
        self.backend = backend
        self.buffer = buffer or LastFrameBuffer()
        self.stop_event = threading.Event()
        self.read_failures = 0
        self.last_error = ''
    def run(self):
        cap_flags = 0 if self.backend is None else self.backend
        cap = cv2.VideoCapture(self.src, cap_flags) if cap_flags else cv2.VideoCapture(self.src)
        if self.width: cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.width)
        if self.height: cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.height)
        while not self.stop_event.is_set():
            ok, frame = cap.read()
            if not ok:
                self.read_failures += 1
                self.last_error = 'capture read failed'
                time.sleep(0.01)
                continue
            self.buffer.put(frame)
        cap.release()
    def stop(self):
        self.stop_event.set()

# Load ONNX and prepare session
onnx_path = str(onnx_path) if 'onnx_path' in globals() else 'runs/weights/best.onnx'
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if ort.get_device() == 'GPU' else ['CPUExecutionProvider']
sess = ort.InferenceSession(onnx_path, providers=providers)
input_name = sess.get_inputs()[0].name

# Simple letterbox preprocess
def letterbox(img, new_shape=640, color=(114,114,114)):
    shape = img.shape[:2]  # H, W
    if isinstance(new_shape, int):
        r = float(new_shape) / max(shape)
    else:
        r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))
    dw, dh = new_shape - new_unpad[0], new_shape - new_unpad[1]
    dw //= 2; dh //= 2
    img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    img = cv2.copyMakeBorder(img, dh, dh, dw, dw, cv2.BORDER_CONSTANT, value=color)
    return img, r, (dw, dh)

# Minimal NMS

def nms(boxes, scores, iou=0.45):
    idxs = scores.argsort()[::-1]
    keep = []
    while len(idxs):
        i = idxs[0]
        keep.append(i)
        if len(idxs) == 1: break
        iou_vals = iou_with(boxes[i], boxes[idxs[1:]])
        idxs = idxs[1:][iou_vals < iou]
    return keep

def iou_with(box, boxes):
    x1 = np.maximum(box[0], boxes[:,0]); y1 = np.maximum(box[1], boxes[:,1])
    x2 = np.minimum(box[2], boxes[:,2]); y2 = np.minimum(box[3], boxes[:,3])
    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    area1 = (box[2]-box[0]) * (box[3]-box[1])
    area2 = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
    union = area1 + area2 - inter + 1e-6
    return inter / union

# Runtime loop (press q to quit)
buffer = LastFrameBuffer()
grabber = AsyncGrabber(0, backend=cv2.CAP_DSHOW, buffer=buffer)
grabber.start()

conf_thres = 0.3
input_size = 640

try:
    while True:
        frame = buffer.get()
        if frame is None:
            time.sleep(0.01)
            continue
        img, r, (dw, dh) = letterbox(frame, input_size)
        im = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        im = np.transpose(im, (2,0,1))[None]
        preds = sess.run(None, {input_name: im})[0]
        boxes_xywh = preds[..., :4]
        scores = preds[..., 4:]
        confs = scores.max(-1)
        cls = scores.argmax(-1)

        mask = confs > conf_thres
        if mask.any():
            boxes_xywh = boxes_xywh[mask]
            confs = confs[mask]
            cls = cls[mask]
            # Convert xywh to xyxy on letterboxed image
            xyxy = np.zeros_like(boxes_xywh)
            xyxy[:,0] = boxes_xywh[:,0] - boxes_xywh[:,2]/2
            xyxy[:,1] = boxes_xywh[:,1] - boxes_xywh[:,3]/2
            xyxy[:,2] = boxes_xywh[:,0] + boxes_xywh[:,2]/2
            xyxy[:,3] = boxes_xywh[:,1] + boxes_xywh[:,3]/2
            # NMS per class
            final = []
            for c in np.unique(cls):
                m = cls == c
                keep = nms(xyxy[m], confs[m], iou=0.50)
                for k in keep:
                    final.append((xyxy[m][k], int(c), float(confs[m][k])))
            # Draw
            for (x1,y1,x2,y2), c, s in final:
                # map back to original frame
                x1 = (x1 - dw) / r; y1 = (y1 - dh) / r
                x2 = (x2 - dw) / r; y2 = (y2 - dh) / r
                p1 = (int(max(0,x1)), int(max(0,y1)))
                p2 = (int(min(frame.shape[1]-1,x2)), int(min(frame.shape[0]-1,y2)))
                cv2.rectangle(frame, p1, p2, (0,255,0), 2)
                cv2.putText(frame, f'{c}:{s:.2f}', (p1[0], max(0,p1[1]-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1, cv2.LINE_AA)
        cv2.imshow('ONNX Edge Runtime', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    grabber.stop()
    grabber.join(timeout=1.0)
    cv2.destroyAllWindows()

In [ ]:
# Section 8: Adaptive throttling and FPS meter
from collections import deque

class FpsMeter:
    def __init__(self, window=50):
        self.ts = deque(maxlen=window)
    def tick(self):
        self.ts.append(time.time())
    def fps(self):
        if len(self.ts) < 2:
            return 0.0
        dt = self.ts[-1] - self.ts[0]
        return (len(self.ts)-1) / dt if dt > 0 else 0.0

target_fps = 25.0
meter = FpsMeter(60)

# Example: integrate into Section 7 loop by sleeping to cap FPS
# Add near end of the while True loop in Section 7:
print('Tip: integrate adaptive sleep into runtime loop:')
print('    meter.tick(); cur = meter.fps();')
print('    if cur > target_fps: time.sleep(max(0.0, 1.0/target_fps))')

In [ ]:
# Section 9: Backend abstraction (PyTorch/ONNX/TensorRT)
from dataclasses import dataclass

@dataclass
class BackendConfig:
    name: str  # 'onnx' | 'torch' | 'tensorrt'
    weights: str
    device: str = 'cpu'

class DetectorBackend:
    def __init__(self, cfg: BackendConfig):
        self.cfg = cfg
        self.impl = None
        if cfg.name == 'onnx':
            providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if ort.get_device() == 'GPU' else ['CPUExecutionProvider']
            self.impl = ort.InferenceSession(cfg.weights, providers=providers)
            self.kind = 'onnx'
        elif cfg.name == 'torch':
            from ultralytics import YOLO
            self.impl = YOLO(cfg.weights)
            self.kind = 'torch'
        elif cfg.name == 'tensorrt':
            # Placeholder: integrate TensorRT runtime/pycuda here if available
            raise NotImplementedError('TensorRT backend not wired in this notebook cell')
        else:
            raise ValueError(f'Unknown backend: {cfg.name}')

    def infer(self, img_bgr: np.ndarray, input_size=640, conf=0.3):
        if self.kind == 'onnx':
            img, r, (dw, dh) = letterbox(img_bgr, input_size)
            im = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            im = np.transpose(im, (2,0,1))[None]
            out = self.impl.run(None, {self.impl.get_inputs()[0].name: im})[0]
            # parse like Section 7
            boxes_xywh = out[..., :4]
            scores = out[..., 4:]
            confs = scores.max(-1)
            cls = scores.argmax(-1)
            mask = confs > conf
            results = []
            if mask.any():
                b = boxes_xywh[mask]; s = confs[mask]; c = cls[mask]
                xyxy = np.zeros_like(b)
                xyxy[:,0] = b[:,0] - b[:,2]/2; xyxy[:,1] = b[:,1] - b[:,3]/2
                xyxy[:,2] = b[:,0] + b[:,2]/2; xyxy[:,3] = b[:,1] + b[:,3]/2
                for cc in np.unique(c):
                    m = c == cc
                    keep = nms(xyxy[m], s[m], iou=0.5)
                    for k in keep:
                        x1,y1,x2,y2 = xyxy[m][k]
                        x1 = (x1 - dw) / r; y1 = (y1 - dh) / r
                        x2 = (x2 - dw) / r; y2 = (y2 - dh) / r
                        results.append(((x1,y1,x2,y2), int(cc), float(s[m][k])))
            return results
        elif self.kind == 'torch':
            from ultralytics.utils.plotting import Annotator
            res = self.impl.predict(img_bgr, imgsz=input_size, conf=conf, verbose=False)
            results = []
            for r in res:
                if len(r.boxes) == 0:
                    continue
                for b in r.boxes:
                    xyxy = b.xyxy[0].cpu().numpy().tolist()
                    c = int(b.cls[0].item()); s = float(b.conf[0].item())
                    results.append((xyxy, c, s))
            return results
        else:
            raise RuntimeError('Backend not implemented')

# Example usage
# backend = DetectorBackend(BackendConfig('onnx', 'runs/weights/best.onnx'))
# dets = backend.infer(frame)

In [ ]:
# Section 10: Micro-benchmarking
import statistics

def bench_infer(sess, warmup=5, iters=50, size=640):
    dummy = (np.random.rand(size, size, 3) * 255).astype(np.uint8)
    input_name = sess.get_inputs()[0].name
    # warmup
    for _ in range(warmup):
        im = cv2.cvtColor(dummy, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        im = np.transpose(im, (2,0,1))[None]
        sess.run(None, {input_name: im})
    # timed
    ts = []
    for _ in range(iters):
        t0 = time.time()
        im = cv2.cvtColor(dummy, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        im = np.transpose(im, (2,0,1))[None]
        _ = sess.run(None, {input_name: im})
        ts.append(time.time()-t0)
    mean_ms = 1000*statistics.mean(ts)
    p95_ms = 1000*statistics.quantiles(ts, n=20)[-1]
    print(f'ONNXRuntime infer: mean {mean_ms:.2f} ms, p95 {p95_ms:.2f} ms over {iters} iters')

if 'sess' in globals():
    bench_infer(sess)
else:
    print('Skip: ONNX session not available in this kernel state')

In [ ]:
# Section 11: Packaging and artifact summary
from pathlib import Path
artifacts = {
    'trained_pt': Path('runs/detect/train/weights/best.pt'),
    'exported_onnx': Path('runs/weights/best.onnx'),
    'optional_trt': Path('runs/weights/best_fp16.trt'),
}
for name, p in artifacts.items():
    print(f'{name}: {p} -> {"OK" if p.exists() else "MISSING"}')

print('\nTo use ONNX in the app, set YOLO_WEIGHTS to runs/weights/best.onnx in your app config or env.')

In [ ]:
# Section 12: Docker packaging notes (CPU ONNX Runtime)
notes = r'''\
Suggested Dockerfile (CPU, slim):

FROM python:3.10-slim

# System deps (opencv basics)
RUN apt-get update && apt-get install -y --no-install-recommends \\
    libglib2.0-0 libsm6 libxrender1 libxext6 \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

COPY . .
ENV YOLO_WEIGHTS=runs/weights/best.onnx YOLO_BACKEND=onnx

# If your app exposes 5000
EXPOSE 5000
CMD ["python", "src/server.py"]

requirements.txt minimal adds:
- numpy<2
- opencv-python>=4.9.0.80
- onnxruntime>=1.17.0
- ultralytics>=8.2.0
- flask (and your other deps)
'''
print(notes)